# Lab 6 — SHAP Interpretability

**Day 03 · Classification & Model Interpretation · Cisco AI/ML Training**

---

## Learning objectives

1. Compute **SHAP values** to explain individual logistic-regression predictions.
2. Rank features by **mean |SHAP|** across a sample of test rows.
3. Connect SHAP attributions to business intuition (why `int_rate` matters for default risk).
4. Visualize global importance with a SHAP bar plot.

> **Checkpoints:** SHAP shape **(20, 5)** · top driver **`int_rate`** · mean |SHAP| printed per feature

**Companion script:** `../scripts/lab06_shap_interpretability.py`


## What is SHAP?

**SHAP** (SHapley Additive exPlanations) assigns each feature a contribution to a single prediction. For a loan scored as high-risk, SHAP answers: *how much did `int_rate` push the score up vs `annual_inc` pulling it down?*

| Concept | Meaning |
|---------|--------|
| **Base value** | Average model output over the training background |
| **SHAP value** | Feature's push toward higher or lower default probability |
| **Sum** | base + sum(SHAP) ≈ model output for that row |

We use the same **scaled numeric** logistic model as Labs 2–4 (five features). Lab 5 added categoricals in a pipeline; here we keep the explainer simple and fast.

```text
  scale → fit logistic → SHAP Explainer(background) → explain 20 test rows
```


---

## 1. Load data, split, and train


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-03":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

OUTPUT_DIR = GH_ROOT / "hands-on" / "day-03" / "scripts" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

X = df[NUMERIC_FEATURES]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

print(f"train size: {len(X_train)}, test size: {len(X_test)}")
print(f"coefficients (scaled features):")
for name, coef in zip(NUMERIC_FEATURES, model.coef_[0]):
    print(f"  {name}: {coef:+.4f}")


---

## 2. Build SHAP explainer and score 20 test rows

The explainer uses the **training** matrix as background (expected feature distribution). We explain the first 20 scaled test rows.


In [ ]:
sample = X_test_s[:20]
explainer = shap.Explainer(model, X_train_s, feature_names=NUMERIC_FEATURES)
shap_values = explainer(sample)

print(f"SHAP values shape: {shap_values.values.shape}")
print(f"base values (first 3): {shap_values.base_values[:3].round(4)}")


---

## 3. Global ranking — mean |SHAP| per feature


In [ ]:
mean_abs = np.abs(shap_values.values).mean(axis=0)
top_idx = int(np.argmax(mean_abs))
top_feature = NUMERIC_FEATURES[top_idx]

ranking = (
    pd.DataFrame({"feature": NUMERIC_FEATURES, "mean_abs_shap": mean_abs.round(4)})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
display(ranking)
print(f"top driver (mean |SHAP|): {top_feature}")


### Why does `int_rate` often rank first?

Higher interest rates usually reflect **riskier borrowers** or **subprime grades**. The model learns that elevated `int_rate` correlates with charge-off — consistent with the positive logistic coefficient you printed above.

SHAP goes further than coefficients: it shows **per-loan** pushes, accounting for interactions with other features on each row.


---

## 4. SHAP bar plot (global importance)


In [ ]:
shap.plots.bar(shap_values, show=False)
bar_plot = OUTPUT_DIR / "shap_bar.png"
plt.tight_layout()
plt.savefig(bar_plot, dpi=120, bbox_inches="tight")
plt.show()
print(f"saved: {bar_plot}")


---

## 5. Single prediction — waterfall for row 0


In [ ]:
shap.plots.waterfall(shap_values[0], show=False)
waterfall_plot = OUTPUT_DIR / "shap_waterfall_row0.png"
plt.tight_layout()
plt.savefig(waterfall_plot, dpi=120, bbox_inches="tight")
plt.show()
print(f"saved: {waterfall_plot}")
print(f"actual default label (row 0): {y_test.iloc[0]}")
print(f"predicted class: {model.predict(sample[:1])[0]}")


---

## 6. Checkpoint summary


In [ ]:
assert shap_values.values.shape == (20, 5)
assert top_feature == "int_rate"
assert mean_abs[top_idx] > mean_abs.mean()
assert bar_plot.is_file()
print("✓ All checkpoint assertions passed")


## Extension — SHAP beeswarm (global pattern)

<!-- cisco-enrich-2026-06 -->

In [ ]:
shap.plots.beeswarm(shap_values, show=False)
beeswarm_plot = OUTPUT_DIR / "shap_beeswarm.png"
plt.tight_layout()
plt.savefig(beeswarm_plot, dpi=120, bbox_inches="tight")
plt.show()
print(f"saved: {beeswarm_plot}")


## Extension — explain a high-risk row

In [ ]:
# row with highest predicted default probability in sample
proba = model.predict_proba(sample)[:, 1]
idx = int(proba.argmax())
shap.plots.waterfall(shap_values[idx], show=False)
plt.tight_layout()
plt.show()
print(f"explained row index {idx}, P(default)={proba[idx]:.3f}, actual={y_test.iloc[idx]}")


---

## Reflection questions

1. How is mean |SHAP| different from the absolute value of logistic coefficients?
2. Why do we pass `X_train_s` as the explainer background instead of `X_test_s`?
3. When would SHAP on a **pipeline** (Lab 5) be harder than on a plain scaled matrix?

**Previous:** [Lab 5 — sklearn Pipeline](lab05_sklearn_pipeline.ipynb)  
**Day 03 complete.** Next: Day 04 — Distance, KNN & MLflow (`../day-04/README.md`)
